# ACOS IndoBERT V5 — Full Experiment Pipeline (GPU AMD MI300X)

> **Notebook ini dirancang khusus untuk GPU enterprise (AMD MI300X 191 GB VRAM).**
> Menjalankan semua kombinasi eksperimen secara otomatis dengan monitoring hardware lengkap.

| Item | Nilai |
|------|-------|
| **GPU Target** | AMD Instinct MI300X VF — 191.69 GB VRAM |
| **Early Stopping** | DINONAKTIFKAN (PATIENCE = 0) |
| **Epochs** | 50 / 75 / 100 |
| **Split Ratio** | 80:10:10 / 70:15:15 / 60:20:20 |
| **Cross Validation** | 5-Fold dan 10-Fold (berbasis review_id) |
| **Total Run** | 54 (serial, otomatis) |
| **Monitoring** | VRAM, waktu per epoch, ROCm stats, hardware log |

---

## Alur Eksekusi

```
Sel 1  : Diagnostik GPU & Hardware (wajib pertama)
Sel 2  : Konfigurasi (PATIENCE=0, batch besar, AMP)
Sel 3  : Import & path setup (dua-root architecture)
Sel 4  : HardwareMonitor — kelas monitoring real-time
Sel 5  : Backbone IndoBERT & tokenizer
Sel 6  : ExperimentGrid — preview 54 run (dry_run)
Sel 7  : Persiapan data semua split + fold (1x, cached)
Sel 8  : Training adapter function
Sel 9  : Run semua eksperimen (otomatis, 1-per-1)
Sel 10 : Agregasi & perbandingan hasil
Sel 11 : Hardware summary per run
```

## Sel 1 — Diagnostik GPU & Hardware

In [1]:
# Sel 1: Diagnostik GPU & Hardware (AMD MI300X / ROCm)
# Wajib dijalankan pertama. Mendeteksi platform dan memvalidasi
# bahwa GPU enterprise tersedia dan VRAM mencukupi.
import subprocess, sys, os, time, platform, json
from datetime import datetime

SESSION_START = datetime.now()
SESSION_START_TS = SESSION_START.strftime('%d%m%Y_%H%M%S')
print(f'Waktu mulai sesi : {SESSION_START.strftime("%Y-%m-%d %H:%M:%S")}')
print(f'Python           : {sys.version}')
print(f'Platform         : {platform.platform()}')
print()

import torch
HAS_CUDA = torch.cuda.is_available()
HAS_ROCM = HAS_CUDA and 'rocm' in torch.__version__.lower()
DEVICE   = 'cuda' if HAS_CUDA else 'cpu'

print(f'PyTorch version  : {torch.__version__}')
print(f'CUDA available   : {HAS_CUDA}')
print(f'ROCm/HIP         : {HAS_ROCM}')
print(f'Device           : {DEVICE}')
print()

if HAS_CUDA:
    n_gpu = torch.cuda.device_count()
    print(f'GPU count        : {n_gpu}')
    for i in range(n_gpu):
        props = torch.cuda.get_device_properties(i)
        vram_gb = props.total_memory / 1024**3
        print(f'  GPU[{i}] : {props.name}')
        print(f'          VRAM = {vram_gb:.2f} GB ({props.total_memory:,} bytes)')
        print(f'          SM   = {props.multi_processor_count} multiprocessors')
    print()

# ROCm-SMI diagnostik
ROCM_SMI_OK = False
try:
    rocm_out = subprocess.check_output(
        ['rocm-smi', '--showmeminfo', 'vram', '--json'],
        stderr=subprocess.DEVNULL, timeout=10, text=True)
    rocm_data = json.loads(rocm_out)
    print('ROCm-SMI VRAM Info:')
    for card, info in rocm_data.items():
        used  = int(info.get('VRAM Total Used Memory (B)', 0)) / 1024**3
        total = int(info.get('VRAM Total Memory (B)', 0)) / 1024**3
        print(f'  {card}: {used:.2f} / {total:.2f} GB ({used/total*100:.1f}% terpakai)')
    ROCM_SMI_OK = True
except Exception:
    print('rocm-smi tidak tersedia - monitoring VRAM via torch.cuda')

MIN_VRAM_GB = 16.0
if HAS_CUDA:
    props = torch.cuda.get_device_properties(0)
    TOTAL_VRAM_GB = props.total_memory / 1024**3
    IS_LARGE_GPU  = TOTAL_VRAM_GB >= MIN_VRAM_GB
    GPU_NAME      = props.name
else:
    TOTAL_VRAM_GB = 0.0
    IS_LARGE_GPU  = False
    GPU_NAME      = 'CPU'

print()
print('=' * 65)
print(f'GPU Name         : {GPU_NAME}')
print(f'Total VRAM       : {TOTAL_VRAM_GB:.2f} GB')
print(f'IS_LARGE_GPU     : {IS_LARGE_GPU}')
print(f'ROCm-SMI         : {ROCM_SMI_OK}')
if not IS_LARGE_GPU:
    print('PERINGATAN: VRAM < 16 GB — notebook ini untuk GPU besar!')
print('=' * 65)

# Simpan info hardware awal
_hw_info = {
    'session_start': SESSION_START.isoformat(),
    'session_ts': SESSION_START_TS,
    'gpu_name': GPU_NAME,
    'total_vram_gb': TOTAL_VRAM_GB,
    'has_rocm': HAS_ROCM,
    'rocm_smi_ok': ROCM_SMI_OK,
    'torch_version': torch.__version__,
    'python_version': sys.version,
    'platform': platform.platform(),
}
print(f'\nHardware info dicatat: {len(_hw_info)} field')

Waktu mulai sesi : 2026-09-24 14:56:40
Python           : 3.12.3 (main, Jul 15 2026, 23:46:41) [GCC 13.3.0]
Platform         : Linux-6.8.0-142-generic-x86_64-with-glibc2.39

PyTorch version  : 2.12.0+rocm7.14.0
CUDA available   : True
ROCm/HIP         : True
Device           : cuda

GPU count        : 1
  GPU[0] : AMD Instinct MI300X VF
          VRAM = 191.69 GB (205,822,885,888 bytes)
          SM   = 304 multiprocessors

ROCm-SMI VRAM Info:
  card0: 12.03 / 191.69 GB (6.3% terpakai)

GPU Name         : AMD Instinct MI300X VF
Total VRAM       : 191.69 GB
IS_LARGE_GPU     : True
ROCm-SMI         : True

Hardware info dicatat: 9 field


## Sel 2 — Konfigurasi (Dioptimalkan untuk AMD MI300X)

> **Perubahan kunci vs notebook V4:**
> - `PATIENCE = 0` — no early stopping (V4: PATIENCE=5)
> - `STEP1_BATCH_SIZE = 96` — 4x lebih besar (V4: 24)
> - `STEP2_BATCH_SIZE = 64` — 4x lebih besar (V4: 16)
> - `RESUME_LAST_SESSION = False` — folder baru tiap run
> - `USE_AMP = True` — bfloat16 mixed precision

In [2]:
# Sel 2: Konfigurasi Master — AMD MI300X Edition

# Domain & backbone
DOMAIN   = 'appsid'
BACKBONE = 'indobert'

# ======================================================
# DIMENSI EKSPERIMEN — edit sesuai kebutuhan
# ======================================================
EXPERIMENT_EPOCHS  = [50, 75, 100]
EXPERIMENT_RATIOS  = [
    (0.8, 0.1),   # 80:10:10
    (0.7, 0.15),  # 70:15:15
    (0.6, 0.2),   # 60:20:20
]
EXPERIMENT_CV = [
    {'n_splits': 5},
    {'n_splits': 10},
]

# ======================================================
# HYPERPARAMETER TRAINING — MI300X optimized
# ======================================================
MAX_SEQ_LENGTH     = 128
STEP1_BATCH_SIZE   = 96    # V4: 24  — 4x untuk VRAM 191 GB
STEP2_BATCH_SIZE   = 64    # V4: 16  — 4x
STEP1_LR           = 2e-5
STEP2_LR           = 5e-5
SEED               = 42
DO_LOWER_CASE      = True

# Early stopping: NONAKTIF untuk GPU besar
PATIENCE               = 0    # 0 = DINONAKTIFKAN (V4: 5)
MIN_EPOCHS_BEFORE_STOP = 5    # tidak relevan bila PATIENCE=0

# Session control
RESUME_LAST_SESSION = False   # selalu buat folder baru

# ======================================================
# ROCm / AMD-specific optimizations
# ======================================================
ROCM_BENCHMARK   = True      # torch.backends.cudnn.benchmark
USE_AMP          = True      # Automatic Mixed Precision
AMP_DTYPE        = 'bfloat16'  # bf16 lebih stabil dari fp16 di MI300X
GRAD_ACCUM_STEPS = 1         # gradient accumulation steps
NUM_WORKERS      = 4         # DataLoader workers

# ======================================================
# MONITORING & LOGGING
# ======================================================
LOG_EVERY_N_STEPS  = 10      # cetak stats setiap N batch
MONITOR_VRAM       = True    # catat VRAM per epoch
SAVE_HARDWARE_LOG  = True    # simpan hardware_log.json
VRAM_FLUSH_EPOCH   = True    # torch.cuda.empty_cache() tiap epoch

# Print ringkasan
print('=' * 65)
print('KONFIGURASI AMD MI300X V5')
print('=' * 65)
print(f'Domain              : {DOMAIN}')
print(f'Backbone            : {BACKBONE}')
print(f'Experiment epochs   : {EXPERIMENT_EPOCHS}')
print(f'Experiment ratios   : {[(int(r[0]*100), int(r[1]*100)) for r in EXPERIMENT_RATIOS]}')
print(f'Experiment CV       : {[c["n_splits"] for c in EXPERIMENT_CV]}-fold')
total_runs = (len(EXPERIMENT_EPOCHS) * len(EXPERIMENT_RATIOS) +
              sum(len(EXPERIMENT_EPOCHS) * c['n_splits'] for c in EXPERIMENT_CV))
print(f'Total runs          : {total_runs}')
print()
print(f'STEP1_BATCH_SIZE    : {STEP1_BATCH_SIZE} (V4: 24)')
print(f'STEP2_BATCH_SIZE    : {STEP2_BATCH_SIZE} (V4: 16)')
print(f'PATIENCE            : {PATIENCE} (0 = no early stop)')
print(f'RESUME_LAST_SESSION : {RESUME_LAST_SESSION}')
print(f'USE_AMP             : {USE_AMP} ({AMP_DTYPE})')
print(f'ROCM_BENCHMARK      : {ROCM_BENCHMARK}')
print('=' * 65)

KONFIGURASI AMD MI300X V5
Domain              : appsid
Backbone            : indobert
Experiment epochs   : [50, 75, 100]
Experiment ratios   : [(80, 10), (70, 15), (60, 20)]
Experiment CV       : [5, 10]-fold
Total runs          : 54

STEP1_BATCH_SIZE    : 96 (V4: 24)
STEP2_BATCH_SIZE    : 64 (V4: 16)
PATIENCE            : 0 (0 = no early stop)
RESUME_LAST_SESSION : False
USE_AMP             : True (bfloat16)
ROCM_BENCHMARK      : True


## Sel 3 — Import & Path Setup

In [ ]:
# Sel 3: Import semua modul & setup path dua-root (ROBUST V4 PATTERN)
import os, sys, time, json, pickle, random, math, importlib
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import torch

# ============================================================
# DETEKSI ROOT DAN SETUP PATH (MULTI-TIER DEFENSIVE)
# ============================================================

def _cari_base_project():
    """Cari root proyek yang mengandung Extract-Classify-ACOS dan ACOS-IndoBERT."""
    kandidat = [
        '/shared-docker/ACOS',
        'd:/laragon/www/ACOS-ASLI',
        'D:/laragon/www/ACOS-ASLI',
        str(Path.cwd().parent.parent),
        str(Path.cwd().parent),
        str(Path.cwd()),
    ]
    
    for _c in kandidat:
        p = Path(_c)
        if not p.exists():
            continue
        extract = p / 'Extract-Classify-ACOS'
        indo = p / 'ACOS-IndoBERT'
        # Verifikasi subfolder kunci ada
        if (extract.exists() and (extract / 'modeling.py').exists() and
            indo.exists() and (indo / 'acos_id').is_dir()):
            return str(p.resolve())
    return None


def _cari_indo_root():
    """Cari folder ACOS-IndoBERT dengan subfolder acos_id/."""
    kandidat = []
    _base = globals().get('base_project_dir') or str(Path.cwd())
    kandidat += [
        str(Path(_base) / 'ACOS-IndoBERT'),
        str(Path('ACOS-IndoBERT').absolute()),
        str((Path.cwd().parent / 'ACOS-IndoBERT').absolute()),
        str(Path.cwd()),
    ]
    for _d in kandidat:
        p = Path(_d)
        if p.is_dir() and (p / 'acos_id').is_dir():
            return str(p.resolve())
    return None


base_project_dir = _cari_base_project()
if base_project_dir is None:
    raise RuntimeError(
        'Root proyek tidak ditemukan. Pastikan struktur:\n'
        '  ACOS-ASLI/\n'
        '    Extract-Classify-ACOS/modeling.py\n'
        '    ACOS-IndoBERT/acos_id/'
    )

upstream_root = str((Path(base_project_dir) / 'Extract-Classify-ACOS').resolve())
indo_root = _cari_indo_root()
if indo_root is None:
    raise RuntimeError(
        f'Folder ACOS-IndoBERT/acos_id tidak ditemukan di {base_project_dir}'
    )

# Verifikasi file penting ada
ACOS_ID_MODULES = ['taxonomy', 'checkpoint', 'cross_val', 'experiment_runner', 'model_wrappers']
_acos_id_dir = Path(indo_root) / 'acos_id'
_missing = [f"{m}.py" for m in ACOS_ID_MODULES
            if not (_acos_id_dir / f"{m}.py").is_file()
            or (_acos_id_dir / f"{m}.py").stat().st_size == 0]
if _missing:
    raise RuntimeError(f'acos_id tidak lengkap di {_acos_id_dir}; hilang: {_missing}')

# Clear sys.path dan module cache
def _prepend_path(p):
    """Paksa p ke posisi terdepan sys.path."""
    while p in sys.path:
        sys.path.remove(p)
    sys.path.insert(0, p)

_prepend_path(indo_root)
_prepend_path(upstream_root)

# Clear acos_id cache
for _m in list(sys.modules):
    if _m == 'acos_id' or _m.startswith('acos_id.'):
        del sys.modules[_m]

print(f'✅ Paths configured:')
print(f'  base_project_dir : {base_project_dir}')
print(f'  upstream_root    : {upstream_root}')
print(f'  indo_root        : {indo_root}')

# ============================================================
# IMPORT MODUL
# ============================================================
print(f'\n{"="*60}')
print('Importing modules...')
print("="*60)

from bert_utils.tokenization import BertTokenizer
print('  ✓ BertTokenizer')

from modeling import BertForQuadABSA, CategorySentiClassification
print('  ✓ BertForQuadABSA, CategorySentiClassification')

acos_id = importlib.import_module('acos_id')
print(f'  ✓ acos_id (v{acos_id.__version__})')

acos_taxonomy = importlib.import_module('acos_id.taxonomy')
print(f'  ✓ taxonomy ({len(acos_taxonomy.ASPECT_LABELS)} aspects)')

acos_ckpt = importlib.import_module('acos_id.checkpoint')
print('  ✓ checkpoint')

acos_cross_val = importlib.import_module('acos_id.cross_val')
build_with_ratio = acos_cross_val.build_with_ratio
build_kfold_splits = acos_cross_val.build_kfold_splits
ExperimentGrid = acos_cross_val.ExperimentGrid
tokenize_all_splits = acos_cross_val.tokenize_all_splits
print('  ✓ cross_val')

acos_experiment_runner = importlib.import_module('acos_id.experiment_runner')
prepare_all_data = acos_experiment_runner.prepare_all_data
run_all_experiments = acos_experiment_runner.run_all_experiments
aggregate_cv_results = acos_experiment_runner.aggregate_cv_results
print('  ✓ experiment_runner')

model_wrappers = importlib.import_module('acos_id.model_wrappers')
print('  ✓ model_wrappers')

print(f'\n✅ ALL IMPORTS SUCCESSFUL!')

# Seed global
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    if ROCM_BENCHMARK:
        torch.backends.cudnn.benchmark = True

# Path directories
data_dir        = os.path.join(indo_root, 'data', 'Apps-ACOS')
processed_dir   = os.path.join(data_dir, 'processed')
tokenized_base  = os.path.join(indo_root, 'tokenized_data')
results_base    = os.path.join(indo_root, 'results')
experiments_dir = os.path.join(results_base, 'experiments')
bert_cache_dir  = os.path.join(indo_root, 'backbones', 'indobert_base_p1')

os.makedirs(experiments_dir, exist_ok=True)

print(f'\nDirectory paths configured:')
print(f'  data_dir        : {data_dir}')
print(f'  tokenized_base  : {tokenized_base}')
print(f'  experiments_dir : {experiments_dir}')
print(f'\n✅ Setup complete!')


RuntimeError: Root proyek tidak ditemukan. Edit _ROOT_CANDIDATES di Sel 3.

## Sel 4 — HardwareMonitor

Kelas untuk monitoring real-time:
- VRAM (used/total/%) per epoch
- Waktu per epoch dan per batch
- GPU utilization via `rocm-smi` atau `nvidia-smi`
- Menyimpan `hardware_log.json` di setiap folder run

In [ ]:
# Sel 4: HardwareMonitor — monitoring real-time GPU AMD MI300X
import threading, time, subprocess, json
from dataclasses import dataclass, field, asdict
from typing import List, Optional


def _fmt_dur(sec: float) -> str:
    if sec < 60: return f'{sec:.1f}s'
    m, s = divmod(int(sec), 60)
    if m < 60: return f'{m}m{s:02d}s'
    h, m = divmod(m, 60)
    return f'{h}h{m:02d}m'


@dataclass
class EpochStats:
    epoch: int
    phase: str = ''
    run_id: str = ''
    epoch_start: float = 0.0
    epoch_end: float = 0.0
    duration_sec: float = 0.0
    loss: float = 0.0
    f1: float = 0.0
    vram_used_gb: float = 0.0
    vram_total_gb: float = 0.0
    vram_pct: float = 0.0
    gpu_util_pct: Optional[float] = None
    batch_per_sec: float = 0.0
    samples_per_sec: float = 0.0


@dataclass
class RunStats:
    run_id: str
    started_at: str = ''
    finished_at: str = ''
    total_duration_sec: float = 0.0
    config: dict = field(default_factory=dict)
    epochs: List[EpochStats] = field(default_factory=list)
    hardware: dict = field(default_factory=dict)
    metrics: dict = field(default_factory=dict)


class HardwareMonitor:
    """Monitor VRAM, waktu, dan GPU utilization per epoch.

    Usage::
        monitor = HardwareMonitor(log_dir=result_dir)
        monitor.start_run('split_801010_ep50', config=cfg)
        for epoch in range(1, NUM_EPOCHS + 1):
            monitor.start_epoch(epoch, phase='step1')
            # ... training ...
            stats = monitor.end_epoch(loss=0.42, f1=96.5,
                                      n_batches=627, batch_size=96)
        monitor.end_run(metrics={'step1_f1': 97.1})
    """

    def __init__(self, log_dir: str, device: int = 0):
        self.log_dir = log_dir
        self.device  = device
        self._run: Optional[RunStats] = None
        self._epoch_start = 0.0
        self._current_epoch = 0
        self._current_phase = ''
        os.makedirs(log_dir, exist_ok=True)

    def start_run(self, run_id: str, config: dict = None):
        hw = self._collect_hardware()
        self._run = RunStats(
            run_id=run_id,
            started_at=datetime.now().isoformat(),
            config=config or {},
            hardware=hw,
        )
        self._print_header(run_id, hw)

    def start_epoch(self, epoch: int, phase: str = 'step1'):
        self._epoch_start = time.time()
        self._current_epoch = epoch
        self._current_phase = phase

    def end_epoch(self, loss: float, f1: float,
                  n_batches: int = 0, batch_size: int = 96) -> EpochStats:
        t_end = time.time()
        dur   = t_end - self._epoch_start
        vram  = self._get_vram()
        gpu_u = self._get_gpu_util()
        bps   = n_batches / dur if dur > 0 else 0
        sps   = (n_batches * batch_size) / dur if dur > 0 else 0

        stats = EpochStats(
            epoch=self._current_epoch,
            phase=self._current_phase,
            run_id=self._run.run_id if self._run else '',
            epoch_start=self._epoch_start,
            epoch_end=t_end,
            duration_sec=round(dur, 2),
            loss=round(loss, 6),
            f1=round(f1, 4),
            vram_used_gb=vram['used_gb'],
            vram_total_gb=vram['total_gb'],
            vram_pct=vram['pct'],
            gpu_util_pct=gpu_u,
            batch_per_sec=round(bps, 2),
            samples_per_sec=round(sps, 1),
        )
        if self._run:
            self._run.epochs.append(stats)
        self._print_epoch(stats)
        return stats

    def end_run(self, metrics: dict = None):
        if not self._run: return
        self._run.finished_at = datetime.now().isoformat()
        self._run.total_duration_sec = sum(e.duration_sec for e in self._run.epochs)
        self._run.metrics = metrics or {}
        self._save_log()
        self._print_footer()

    def vram_str(self) -> str:
        v = self._get_vram()
        return f"{v['used_gb']:.2f}/{v['total_gb']:.2f} GB ({v['pct']:.1f}%)"

    def _get_vram(self) -> dict:
        if not torch.cuda.is_available():
            return {'used_gb': 0.0, 'total_gb': 0.0, 'pct': 0.0}
        used  = torch.cuda.memory_allocated(self.device)
        total = torch.cuda.get_device_properties(self.device).total_memory
        ug = used / 1024**3
        tg = total / 1024**3
        return {'used_gb': round(ug, 3), 'total_gb': round(tg, 3),
                'pct': round(ug / tg * 100, 2) if tg > 0 else 0.0}

    def _get_gpu_util(self) -> Optional[float]:
        try:
            if HAS_ROCM:
                out = subprocess.check_output(
                    ['rocm-smi', '--showuse', '--json'],
                    stderr=subprocess.DEVNULL, timeout=5, text=True)
                for card, info in json.loads(out).items():
                    u = info.get('GPU use (%)', None)
                    if u is not None: return float(u)
            else:
                out = subprocess.check_output(
                    ['nvidia-smi', '--query-gpu=utilization.gpu',
                     '--format=csv,noheader,nounits'],
                    stderr=subprocess.DEVNULL, timeout=5, text=True)
                return float(out.strip().split('\n')[0])
        except Exception:
            pass
        return None

    def _collect_hardware(self) -> dict:
        hw = {'timestamp': datetime.now().isoformat(),
              'torch': torch.__version__, 'cuda': HAS_CUDA, 'rocm': HAS_ROCM}
        if HAS_CUDA:
            p = torch.cuda.get_device_properties(0)
            hw['gpu_name'] = p.name
            hw['vram_total_gb'] = round(p.total_memory / 1024**3, 3)
            hw['sm_count'] = p.multi_processor_count
        return hw

    def _save_log(self):
        if not self._run: return
        path = os.path.join(self.log_dir, 'hardware_log.json')
        with open(path, 'w', encoding='utf-8') as fh:
            json.dump(asdict(self._run), fh, indent=2, ensure_ascii=False, default=str)

    def _print_header(self, run_id, hw):
        print(f"\n{'='*72}")
        print(f"  RUN    : {run_id}")
        print(f"  GPU    : {hw.get('gpu_name','CPU')} | "
              f"VRAM: {hw.get('vram_total_gb',0):.2f} GB | ROCm: {hw.get('rocm',False)}")
        print(f"  Start  : {self._run.started_at}")
        print(f"{'='*72}")
        print(f"  {'Ep':>4} {'Phase':>6} {'Loss':>9} {'F1%':>7} "
              f"{'VRAM':>14} {'Util%':>6} {'Samp/s':>8} {'Dur':>8}")
        print(f"  {'-'*4} {'-'*6} {'-'*9} {'-'*7} "
              f"{'-'*14} {'-'*6} {'-'*8} {'-'*8}")

    def _print_epoch(self, s: EpochStats):
        vram_str = f'{s.vram_used_gb:.2f}/{s.vram_total_gb:.2f}G'
        util_str = f'{s.gpu_util_pct:.0f}%' if s.gpu_util_pct is not None else 'N/A'
        print(f"  {s.epoch:>4} {s.phase:>6} {s.loss:>9.4f} {s.f1:>7.2f} "
              f"{vram_str:>14} {util_str:>6} {s.samples_per_sec:>8.0f} {_fmt_dur(s.duration_sec):>8}")

    def _print_footer(self):
        total = self._run.total_duration_sec if self._run else 0
        m = self._run.metrics if self._run else {}
        print(f"  {'='*72}")
        print(f"  Selesai : {self._run.finished_at}")
        print(f"  Durasi  : {_fmt_dur(total)}")
        print(f"  Metrics : Step1 F1={m.get('step1_f1','?')} | Step2 F1={m.get('step2_f1','?')}")
        print(f"{'='*72}\n")


print('HardwareMonitor: OK')
print(f'Contoh: monitor = HardwareMonitor(log_dir=result_dir)')

## Sel 5 — Backbone IndoBERT & Tokenizer

In [ ]:
# Sel 5: Siapkan backbone IndoBERT & tokenizer
from acos_id.checkpoint import prepare_backbone

print('Mempersiapkan backbone IndoBERT...')
backbone_report = prepare_backbone(BACKBONE, bert_cache_dir)
rk = backbone_report.get('rekey', {})
print(f"  Rekey: {rk.get('dilewati', False) and 'dilewati' or 'selesai'}")

tokenizer = BertTokenizer.from_pretrained(bert_cache_dir, do_lower_case=DO_LOWER_CASE)
print(f'  Tokenizer vocab: {len(tokenizer.vocab):,} token')
print()

# Validasi dataset
print('Validasi dataset:')
for split in ('train', 'dev', 'test'):
    fpath = os.path.join(data_dir, f'appsid_quad_{split}.tsv')
    if os.path.exists(fpath):
        n = sum(1 for _ in open(fpath, encoding='utf-8'))
        print(f'  {split:5s}: {n:7,} baris  [OK]')
    else:
        print(f'  {split:5s}: MISSING! {fpath}')

# Validasi backbone files
print()
print('Validasi backbone files:')
for f in ('config.json', 'pytorch_model.bin', 'vocab.txt'):
    fpath = os.path.join(bert_cache_dir, f)
    exists = os.path.exists(fpath)
    sz = os.path.getsize(fpath) / 1024**2 if exists else 0
    print(f'  {f:25s}: {"OK" if exists else "MISSING"} ({sz:.1f} MB)')

print()
print('Backbone & tokenizer: OK')

## Sel 6 — ExperimentGrid (Preview 54 Run)

Jalankan sel ini untuk **melihat semua rencana eksperimen tanpa training**.
Hanya tampilan — tidak ada data yang diproses.

In [ ]:
# Sel 6: Preview semua kombinasi eksperimen
grid = ExperimentGrid()
grid.add_epochs(EXPERIMENT_EPOCHS)
grid.add_ratios(EXPERIMENT_RATIOS)
for cv_cfg in EXPERIMENT_CV:
    grid.add_cv(n_splits=cv_cfg['n_splits'])

grid.print_summary()
print(f'\nTotal: {len(grid)} run')
n_ratio = len(EXPERIMENT_EPOCHS) * len(EXPERIMENT_RATIOS)
n_cv    = sum(len(EXPERIMENT_EPOCHS) * c['n_splits'] for c in EXPERIMENT_CV)
print(f'  Split-ratio: {n_ratio} run')
print(f'  CV total   : {n_cv} run')
print()
# Estimasi kasar: 50 epoch sekitar 4 jam per run di MI300X
est_hr_50  = len(grid) * 4
print(f'Estimasi durasi (semua, 50 ep per run ~4 jam): {est_hr_50} jam = {est_hr_50/24:.1f} hari')
print()
print('Tips: Gunakan mode="ratio" atau mode="cv" di Sel 9 untuk subset.')

## Sel 7 — Persiapan Data (Semua Split + Fold, 1x)

> **Jalankan 1x saja.** Hasilnya di-cache.
> Pemanggilan berikutnya skip file yang sudah ada (`force_rebuild=False`).

**Waktu estimasi: 30–60 menit (1x)**

In [ ]:
# Sel 7: Build semua TSV mentah + tokenisasi (1x, cached)
t0 = time.time()

prepared = prepare_all_data(
    indo_root=indo_root,
    tokenizer=tokenizer,
    ratio_list=EXPERIMENT_RATIOS,
    cv_configs=EXPERIMENT_CV,
    seed=SEED,
    force_rebuild=False,   # ganti True untuk rebuild paksa
)

dur = time.time() - t0
print(f'\nTotal waktu persiapan data: {_fmt_dur(dur)}')
print('Semua data siap untuk training.')

## Sel 8 — Training Adapter Function

Fungsi `train_one_run(cfg)` adalah adapter antara `ExperimentGrid`
dan training loop Step 1 + Step 2.
Setiap panggilan menjalankan 1 kombinasi eksperimen dari awal.

> **Catatan:** Sel ini mendefinisikan fungsi — belum menjalankan training.
> Training dimulai di Sel 9.

In [ ]:
# Sel 8: Adapter — 1 run eksperimen = Step 1 + Step 2
from acos_id.session import session_dirs_from_root
from torch.utils.data import DataLoader


def train_one_run(cfg: dict) -> dict:
    """Jalankan 1 run lengkap (Step 1 + Step 2) dari ExperimentGrid.

    Parameter cfg (dari ExperimentGrid):
      cfg['epochs']        = NUM_EPOCHS untuk run ini
      cfg['tokenized_dir'] = folder tokenized yang dipakai
      cfg['result_dir']    = folder output run ini
      cfg['run_id']        = nama unik run ini
      cfg['seed']          = random seed
    """
    run_id  = cfg['run_id']
    num_ep  = cfg['epochs']
    tok_dir = cfg['tokenized_dir']
    res_dir = cfg['result_dir']
    seed    = cfg.get('seed', SEED)

    if not os.path.isdir(tok_dir):
        raise FileNotFoundError(f'tokenized_dir tidak ada: {tok_dir}')

    # Setup session dirs
    sess = session_dirs_from_root(res_dir)

    # Hardware monitor
    monitor = HardwareMonitor(log_dir=res_dir, device=0)
    monitor.start_run(run_id, config=cfg)

    # AMP scaler
    scaler = None
    if USE_AMP and HAS_CUDA:
        scaler = torch.cuda.amp.GradScaler()

    # ================================================================
    # STEP 1: BertForQuadABSA (co-extraction aspek & opini)
    # ================================================================
    def load_loader(split, batch_size, shuffle):
        path = os.path.join(tok_dir, f'appsid_{split}_quad_bert.tsv')
        dataset = model_wrappers.load_quad_tsv_dataset(
            path, tokenizer, max_seq_length=MAX_SEQ_LENGTH)
        return DataLoader(dataset, batch_size=batch_size,
                          shuffle=shuffle, num_workers=NUM_WORKERS)

    train_loader = load_loader('train', STEP1_BATCH_SIZE, True)
    dev_loader   = load_loader('dev',   STEP1_BATCH_SIZE, False)

    model = BertForQuadABSA.from_pretrained(
        bert_cache_dir,
        num_aspect_labels=len(acos_taxonomy.ASPECT_LABELS),
        num_opinion_labels=len(acos_taxonomy.OPINION_LABELS),
    ).to(DEVICE)

    optimizer = torch.optim.AdamW(model.parameters(), lr=STEP1_LR)

    best_f1_s1 = 0.0
    best_ep_s1 = 0

    print(f'\n  [Step 1] {num_ep} epoch | batch={STEP1_BATCH_SIZE} | '
          f'AMP={USE_AMP}({AMP_DTYPE}) | PATIENCE={PATIENCE}')

    for epoch in range(1, num_ep + 1):
        monitor.start_epoch(epoch, phase='step1')
        model.train()
        total_loss = 0.0
        n_batches  = 0
        t_ep = time.time()

        for step, batch in enumerate(train_loader):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            optimizer.zero_grad()

            if USE_AMP and scaler:
                dtype = torch.bfloat16 if AMP_DTYPE == 'bfloat16' else torch.float16
                with torch.cuda.amp.autocast(dtype=dtype):
                    outputs = model(**batch)
                    loss = outputs['loss'] / GRAD_ACCUM_STEPS
                scaler.scale(loss).backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
            else:
                outputs = model(**batch)
                loss = outputs['loss'] / GRAD_ACCUM_STEPS
                loss.backward()
                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()

            total_loss += loss.item() * GRAD_ACCUM_STEPS
            n_batches  += 1

            if LOG_EVERY_N_STEPS > 0 and (step + 1) % LOG_EVERY_N_STEPS == 0:
                elapsed = time.time() - t_ep
                sps = (n_batches * STEP1_BATCH_SIZE) / elapsed
                print(f'    ep{epoch:3d} step{step+1:4d}/{len(train_loader)} '
                      f'loss={loss.item():.4f} VRAM={monitor.vram_str()} '
                      f'samp/s={sps:.0f}', end='\r')

        # Evaluasi dev
        eval_results = model_wrappers.compute_extraction_metrics(
            model, dev_loader, DEVICE)
        f1 = eval_results['f1']
        prec = eval_results['precision']
        rec = eval_results['recall']

        avg_loss = total_loss / max(n_batches, 1)
        print()  # newline setelah progress bar
        monitor.end_epoch(avg_loss, f1, n_batches, STEP1_BATCH_SIZE)

        if f1 > best_f1_s1:
            best_f1_s1 = f1
            best_ep_s1 = epoch
            ckpt_path = os.path.join(sess['checkpoints'], 'step1_best')
            model.save_pretrained(ckpt_path)
            print(f'    >> Best checkpoint: ep{epoch} F1={f1:.2f}%')

        # PATIENCE = 0 di notebook ini — early stopping TIDAK AKTIF
        # Blok ini tetap ada untuk kompatibilitas tapi tidak pernah terpicu
        if PATIENCE > 0:
            epochs_since_best = epoch - best_ep_s1
            if epoch >= MIN_EPOCHS_BEFORE_STOP and epochs_since_best >= PATIENCE:
                print(f'    >> Early stop ep{epoch} (tidak terpicu karena PATIENCE=0)')
                break

        if VRAM_FLUSH_EPOCH:
            torch.cuda.empty_cache()

    print(f'\n  [Step 1] SELESAI | Best F1={best_f1_s1:.2f}% di ep{best_ep_s1}')

    # ================================================================
    # STEP 2: Klasifikasi Category + Sentiment
    # (Implementasi penuh di sini — serupa Step 1 tapi pakai pair.tsv)
    # ================================================================
    best_f1_s2 = 0.0
    # TODO: Implementasi penuh Step 2
    # Prinsip: ganti load_loader ke pair.tsv, model ke ClassificationModel
    print(f'  [Step 2] TODO — best_f1_s2={best_f1_s2:.2f}%')

    monitor.end_run(metrics={
        'step1_f1': round(best_f1_s1, 4),
        'step1_best_epoch': best_ep_s1,
        'step2_f1': round(best_f1_s2, 4),
    })

    torch.cuda.empty_cache()
    return {'step1_f1': best_f1_s1, 'step2_f1': best_f1_s2}


print('train_one_run: terdefinisi.')
print('Fungsi siap. Training dimulai di Sel 9.')

## Sel 9 — Run Semua Eksperimen (Otomatis, 1-per-1)

> **Estimasi total: 5–8 hari untuk 54 run (50–100 epoch per run)**

Parameter `mode`:
- `'all'` — semua 54 run
- `'ratio'` — hanya 9 run split-ratio
- `'cv'` — hanya 45 run cross-validation

Set `DRY_RUN = True` untuk preview urutan tanpa training.

In [ ]:
# Sel 9: Jalankan semua eksperimen otomatis (1-per-1)

# ======================================================
# KONTROL — edit sesuai kebutuhan
# ======================================================
DRY_RUN      = True    # ganti False untuk training sungguhan
RUN_MODE     = 'all'   # 'all', 'ratio', atau 'cv'
RUN_EPOCHS   = EXPERIMENT_EPOCHS    # atau subset: [50]
# ======================================================

t_start = time.time()

all_results = run_all_experiments(
    train_fn=train_one_run,
    indo_root=indo_root,
    epochs_list=RUN_EPOCHS,
    ratio_list=EXPERIMENT_RATIOS,
    cv_configs=EXPERIMENT_CV,
    seed=SEED,
    mode=RUN_MODE,
    dry_run=DRY_RUN,
)

total_dur = time.time() - t_start
print(f'\nWaktu total: {_fmt_dur(total_dur)}')
print(f'Status: {"DRY RUN" if DRY_RUN else f"{len(all_results)} run selesai"}')

## Sel 10 — Agregasi & Perbandingan Hasil

In [ ]:
# Sel 10: Ringkasan & perbandingan semua hasil
if not all_results:
    print('Belum ada hasil. Jalankan Sel 9 dengan DRY_RUN=False.')
else:
    ok = [r for r in all_results if r.get('status') == 'OK']
    err  = [r for r in all_results if r.get('status') == 'ERROR']
    skip = [r for r in all_results if r.get('status') == 'SKIP_NO_DATA']
    print(f'Total run: {len(all_results)} | OK: {len(ok)} | Error: {len(err)} | Skip: {len(skip)}')

    if ok:
        rows = []
        for r in ok:
            m = r.get('metrics', {})
            split_str = (
                f"{int(r.get('train_ratio',0)*100)}:{int(r.get('dev_ratio',0)*100)}"
                if r['type'] == 'ratio'
                else f"cv{r.get('n_splits','?')}-f{r.get('fold_idx','?')}"
            )
            rows.append({
                'run_id'    : r['run_id'],
                'type'      : r['type'],
                'epochs'    : r['epochs'],
                'split'     : split_str,
                'step1_f1'  : m.get('step1_f1', 0),
                'step2_f1'  : m.get('step2_f1', 0),
                'dur_min'   : round(r.get('duration_sec', 0) / 60, 1),
            })

        df = pd.DataFrame(rows).sort_values('step1_f1', ascending=False)
        print('\nTop 10 berdasarkan Step1 F1:')
        print(df.head(10).to_string(index=False))

        # Agregasi CV
        for n_splits in [5, 10]:
            agg = aggregate_cv_results(all_results, n_splits=n_splits)
            cv_key = f'cv_{n_splits}fold'
            if agg.get(cv_key):
                print(f'\nCV {n_splits}-Fold — mean +/- std:')
                df_cv = pd.DataFrame(agg[cv_key].values())
                cols = ['epoch', 'step1_f1_mean', 'step1_f1_std',
                        'step2_f1_mean', 'step2_f1_std', 'n_folds_completed']
                print(df_cv[[c for c in cols if c in df_cv.columns]].to_string(index=False))

        # Simpan tabel
        summary_path = os.path.join(experiments_dir, 'final_results_summary.csv')
        df.to_csv(summary_path, index=False)
        print(f'\nRingkasan disimpan: {summary_path}')

## Sel 11 — Hardware Summary per Run

Baca semua `hardware_log.json` dan tampilkan tabel penggunaan resource:
VRAM max, VRAM rata-rata, GPU utilization, samples/sec, durasi.

In [ ]:
# Sel 11: Ringkasan hardware semua run
import glob

hw_logs = sorted(glob.glob(os.path.join(experiments_dir, '*', 'hardware_log.json')))
print(f'Hardware logs ditemukan: {len(hw_logs)}')

hw_rows = []
for log_path in hw_logs:
    try:
        with open(log_path, encoding='utf-8') as fh:
            data = json.load(fh)
        eps = data.get('epochs', [])
        if eps:
            vram_vals = [e['vram_used_gb'] for e in eps if e.get('vram_used_gb', 0) > 0]
            util_vals = [e['gpu_util_pct'] for e in eps if e.get('gpu_util_pct') is not None]
            sps_vals  = [e['samples_per_sec'] for e in eps if e.get('samples_per_sec', 0) > 0]
            hw_rows.append({
                'run_id'       : data.get('run_id', '?'),
                'dur_min'      : round(data.get('total_duration_sec', 0) / 60, 1),
                'epochs_done'  : len(eps),
                'vram_max_gb'  : round(max(vram_vals), 3) if vram_vals else 0,
                'vram_avg_gb'  : round(sum(vram_vals) / len(vram_vals), 3) if vram_vals else 0,
                'vram_pct_avg' : round(sum(e['vram_pct'] for e in eps) / len(eps), 2) if eps else 0,
                'gpu_util_avg' : round(sum(util_vals) / len(util_vals), 1) if util_vals else None,
                'samp_s_avg'   : round(sum(sps_vals) / len(sps_vals), 0) if sps_vals else 0,
                'step1_f1'     : data.get('metrics', {}).get('step1_f1', None),
            })
    except Exception as e:
        print(f'  Gagal baca {log_path}: {e}')

if hw_rows:
    df_hw = pd.DataFrame(hw_rows).sort_values('step1_f1', ascending=False)
    print('\nHardware Usage per Run:')
    print(df_hw.to_string(index=False))

    hw_summary_path = os.path.join(experiments_dir, 'hardware_summary.csv')
    df_hw.to_csv(hw_summary_path, index=False)
    print(f'\nHardware summary disimpan: {hw_summary_path}')
else:
    print('Belum ada hardware log (jalankan training terlebih dahulu).')

# Ringkasan sesi total
SESSION_END = datetime.now()
print(f'\nSesi: {SESSION_START.strftime("%Y-%m-%d %H:%M:%S")} s/d {SESSION_END.strftime("%H:%M:%S")}')
print(f'Durasi sesi: {_fmt_dur((SESSION_END - SESSION_START).total_seconds())}')